# Pregnancy Journey Partner Chatbot

## Notebook 04: RAG Pipeline

This notebook builds the Retrieval-Augmented Generation (RAG) pipeline.

Objectives:

- Load the knowledge base
- Retrieve relevant context
- Build prompts
- Generate grounded responses
- Include trusted medical sources
- Apply chatbot safety instructions

In [1]:
from pathlib import Path

import polars as pl
import numpy as np
import chromadb

from sentence_transformers import SentenceTransformer

In [2]:
# Load processed knowledge base

processed_path = Path("../knowledge_base/processed")

chunks_df = pl.read_parquet(
    processed_path / "chunks.parquet"
)

embeddings = np.load(
    processed_path / "embeddings.npy"
)

print("Knowledge base loaded.")

print(f"Chunks: {chunks_df.height}")

Knowledge base loaded.
Chunks: 3596


In [3]:
# Load embedding model

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [4]:
# Connect to ChromaDB

client = chromadb.PersistentClient(
    path="../knowledge_base/vector_db"
)

collection = client.get_collection(
    "pregnancy_knowledge"
)

print("Vector database connected.")

print(
    f"Stored chunks: {collection.count()}"
)

Vector database connected.
Stored chunks: 3596


In [5]:
# Retrieve relevant knowledge

def retrieve_context(question, n_results=5):

    embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=n_results
    )

    return results

In [6]:
# Build context for the LLM

def build_context(results):

    documents = results["documents"][0]

    context = "\n\n".join(documents)

    return context

In [7]:
question = "What are the danger signs during pregnancy?"

results = retrieve_context(question)

context = build_context(results)

print(context[:2000])

after birth.
When to seek care for danger signs
Go to hospital or health centre immediately, day or night, DO NOT wait, if any of the following signs:
 Vaginal bleeding has increased.
 Fits.
 Fast or difficult breathing.
 Fever and too weak to get out of bed.
 Severe headaches with blurred vision.
 Calf pain, redness or swelling; shortness of breath or chest pain.
Go to health centre as soon as possible if any of the following signs:
 Swollen, red or tender breasts or nipples.
 Problems urinating, or leaking.
 Increased pain or infection in the perineum.
 Infection in the area of the wound.
 Smelly vaginal discharge.
INFORMATION AND COUNSELLING SHEETS
M4Care for the mother after birth

WHO recommendations on antenatal care for a positive pregnancy experience
74
D. Interventions for common
physiological symptoms
Background
Women’s bodies undergo substantial changes
during pregnancy, which are brought about by both
hormonal and mechanical effects. These changes lead

In [8]:
# Chatbot system instructions

SYSTEM_PROMPT = """
You are Pregnancy Journey Partner, an educational pregnancy
information assistant.

Your role is to provide clear, simple, supportive and evidence-based
pregnancy information using ONLY the trusted knowledge provided in
the retrieved context.

IMPORTANT RULES:

1. Use the retrieved context as your primary source of information.

2. Do not invent medical facts, recommendations, medications,
   diagnoses, or treatment instructions that are not supported by
   the retrieved context.

3. If the retrieved context does not contain enough information to
   answer a question, clearly say that the available knowledge base
   does not contain enough information rather than making up an answer.

4. For urgent symptoms or danger signs, clearly encourage the user
   to seek appropriate medical care promptly.

5. Do not diagnose the user or claim to replace a doctor, midwife,
   nurse, or other qualified healthcare professional.

6. Answer in simple language that is easy for pregnant women and
   families to understand.

7. When possible, mention the organization and source document used
   to answer the question.

8. Distinguish between general educational information and situations
   that require professional medical assessment.

9. Do not provide false reassurance when a potentially serious
   symptom is described.

10. Never claim that you have examined the user or know their
    individual medical condition.

MEDICAL DISCLAIMER:

Pregnancy Journey Partner provides general educational information
and is not a substitute for professional medical advice, diagnosis,
or treatment. If you have a concerning symptom, are unsure about
your condition, or believe you may be experiencing an emergency,
please contact a qualified healthcare professional or seek
appropriate medical care promptly.
"""

In [9]:
# Build the RAG prompt

def build_rag_prompt(question, context):
    """
    Build the prompt that will be sent to the LLM.

    Parameters
    ----------
    question : str
        User's question.

    context : str
        Relevant information retrieved from the knowledge base.

    Returns
    -------
    str
        Complete prompt for the language model.
    """

    prompt = f"""
{SYSTEM_PROMPT}

RETRIEVED KNOWLEDGE:
--------------------
{context}
--------------------

USER QUESTION:
{question}

INSTRUCTIONS FOR THIS RESPONSE:

- Answer the user's question using the retrieved knowledge.
- Keep the answer clear and easy to understand.
- Do not introduce unsupported medical information.
- If the retrieved knowledge is insufficient, say so honestly.
- Mention the relevant source when appropriate.
- If the question describes a possible danger sign or emergency,
  clearly advise the user to seek appropriate medical care.

Now provide the answer.
"""

    return prompt

In [10]:
# Test the RAG prompt

question = "What are the danger signs during pregnancy?"

results = retrieve_context(
    question,
    n_results=5
)

context = build_context(results)

prompt = build_rag_prompt(
    question,
    context
)

print(prompt[:5000])



You are Pregnancy Journey Partner, an educational pregnancy
information assistant.

Your role is to provide clear, simple, supportive and evidence-based
pregnancy information using ONLY the trusted knowledge provided in
the retrieved context.

IMPORTANT RULES:

1. Use the retrieved context as your primary source of information.

2. Do not invent medical facts, recommendations, medications,
   diagnoses, or treatment instructions that are not supported by
   the retrieved context.

3. If the retrieved context does not contain enough information to
   answer a question, clearly say that the available knowledge base
   does not contain enough information rather than making up an answer.

4. For urgent symptoms or danger signs, clearly encourage the user
   to seek appropriate medical care promptly.

5. Do not diagnose the user or claim to replace a doctor, midwife,
   nurse, or other qualified healthcare professional.

6. Answer in simple language that is easy for pregnant women and
   

In [11]:
# Build source-aware context

def build_source_aware_context(results):
    """
    Combine retrieved chunks with their source information.

    Parameters
    ----------
    results : dict
        Results returned by ChromaDB.

    Returns
    -------
    str
        Context containing medical information and source details.
    """

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    context_parts = []

    for i, (document, metadata) in enumerate(
        zip(documents, metadatas),
        start=1
    ):

        source = f"""
SOURCE {i}
Organization: {metadata['organization']}
Document: {metadata['document']}
Page: {metadata['page_number']}
Title: {metadata['title']}
URL: {metadata['url']}

Content:
{document}
"""

        context_parts.append(source)

    return "\n\n".join(context_parts)

In [12]:
# Test source-aware context

question = "What are the danger signs during pregnancy?"

results = retrieve_context(
    question,
    n_results=5
)

context = build_source_aware_context(results)

print(context[:5000])


SOURCE 1
Organization: WHO
Document: Who pregnancy, childbirth, postpartum
Page: 165
Title: Pregnancy Childbirth Postpartum and Newborn Care: A Guide for Essential Practice
URL: https://www.who.int/publications/b/31363

Content:
after birth.
When to seek care for danger signs
Go to hospital or health centre immediately, day or night, DO NOT wait, if any of the following signs:
 Vaginal bleeding has increased.
 Fits.
 Fast or difficult breathing.
 Fever and too weak to get out of bed.
 Severe headaches with blurred vision.
 Calf pain, redness or swelling; shortness of breath or chest pain.
Go to health centre as soon as possible if any of the following signs:
 Swollen, red or tender breasts or nipples.
 Problems urinating, or leaking.
 Increased pain or infection in the perineum.
 Infection in the area of the wound.
 Smelly vaginal discharge.
INFORMATION AND COUNSELLING SHEETS
M4Care for the mother after birth



SOURCE 2
Organization: WHO
Document: Who recommenda

In [30]:
# ============================================
# Build highly grounded medical RAG prompt
# ============================================

def build_rag_prompt(question, context):

    prompt = f"""
You are Pregnancy Journey Partner, an educational pregnancy
information assistant.

You must answer the user's question using ONLY the retrieved
knowledge below.

IMPORTANT:
The retrieved knowledge is the ONLY source of medical facts
you are allowed to use.

STRICT RULES:

1. Every medical claim in your answer must be directly supported
   by the retrieved knowledge.

2. Do NOT use your general medical knowledge.

3. Do NOT add information that is not explicitly stated in the
   retrieved knowledge.

4. Do NOT infer additional symptoms, causes, treatments,
   time limits, diagnoses, or recommendations.

5. If a detail is not present in the retrieved knowledge,
   leave it out.

6. If the retrieved knowledge is insufficient to answer the
   question, say:

   "The available knowledge base does not contain enough
   information to answer this question."

7. Never diagnose the user.

8. Never prescribe or recommend medication unless the retrieved
   knowledge explicitly provides that information.

9. For danger signs explicitly stated in the retrieved knowledge,
   clearly advise the user to seek appropriate medical care.

10. Keep the answer simple and easy to understand.

11. Do not pretend that information comes from the WHO or UNICEF
    unless that organization and source are included in the
    retrieved context.

12. Do not use information from the original documents that was
    NOT retrieved below.

RETRIEVED KNOWLEDGE
===================

{context}

USER QUESTION
=============

{question}

FINAL INSTRUCTION
=================

Answer using ONLY the retrieved knowledge.

If the retrieved knowledge does not provide enough information,
say so instead of completing the answer from your own knowledge.
"""

    return prompt

In [14]:
# ============================================
# Test the complete prompt
# ============================================

question = "What are the danger signs during pregnancy?"

results = retrieve_context(
    question,
    n_results=5
)

context = build_source_aware_context(results)

prompt = build_rag_prompt(
    question,
    context
)

print(prompt[:7000])



You are Pregnancy Journey Partner, an educational pregnancy
information assistant.

Your role is to provide clear, simple, supportive and evidence-based
pregnancy information using ONLY the trusted knowledge provided in
the retrieved context.

IMPORTANT RULES:

1. Use the retrieved context as your primary source of information.

2. Do not invent medical facts, recommendations, medications,
   diagnoses, or treatment instructions that are not supported by
   the retrieved context.

3. If the retrieved context does not contain enough information to
   answer a question, clearly say that the available knowledge base
   does not contain enough information rather than making up an answer.

4. For urgent symptoms or danger signs, clearly encourage the user
   to seek appropriate medical care promptly.

5. Do not diagnose the user or claim to replace a doctor, midwife,
   nurse, or other qualified healthcare professional.

6. Answer in simple language that is easy for pregnant women and
   

In [15]:
# ============================================
# Load Gemini API key
# ============================================

import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print("Gemini API key loaded successfully.")
else:
    print("Gemini API key not found.")

Gemini API key loaded successfully.


In [16]:
# ============================================
# Initialize Gemini client
# ============================================

from google import genai

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [18]:
# ============================================
# Check available Gemini models
# ============================================

models = client.models.list()

for model in models:
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [20]:
# ============================================
# Check Gemini 3.5 Flash availability
# ============================================

available_model_names = [
    model.name
    for model in client.models.list()
]

if "models/gemini-3.5-flash" in available_model_names:
    print("Gemini 3.5 Flash is available.")
else:
    print("Gemini 3.5 Flash is not available for this API key.")

Gemini 3.5 Flash is available.


In [21]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Say hello to Pregnancy Journey Partner in one sentence."
)

print(response.text)

Hello and welcome to your Pregnancy Journey Partner, where I am here to support, guide, and walk alongside you through every beautiful step of your path to parenthood.


In [32]:
# ============================================
# Generate RAG answer with Gemini
# ============================================

def generate_answer(question, n_results=5):
    """
    Retrieve relevant knowledge and generate an answer
    using Gemini.
    """

    # 1. Retrieve relevant chunks
    results = retrieve_context(
        question,
        n_results=n_results
    )

    # 2. Build source-aware context
    context = build_source_aware_context(results)

    # 3. Build the RAG prompt
    prompt = build_rag_prompt(
        question,
        context
    )

    # 4. Generate answer with Gemini
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    return response.text

In [33]:
# ============================================
# First complete RAG chatbot test
# ============================================

question = "What are the danger signs during pregnancy?"

answer = generate_answer(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

QUESTION:
What are the danger signs during pregnancy?

ANSWER:
According to the World Health Organization (WHO), there are specific danger signs during pregnancy. 

If you experience any of these signs, you must seek appropriate medical care. 

**Go to the hospital or health centre immediately, day or night, WITHOUT waiting, if you have any of the following signs:**
* Vaginal bleeding
* Convulsions
* Severe headaches with blurred vision
* Fever and being too weak to get out of bed
* Severe abdominal pain
* Fast or difficult breathing

**Go to a health centre as soon as possible if you have any of the following signs:**
* Fever
* Abdominal pain
* Feeling ill
* Swelling of the fingers, face, or legs


In [31]:
question = "What are the signs that labour is starting?"

answer = generate_answer(question)

print(answer)

The available knowledge base does not contain enough information to answer this question.
